In [ ]:
DATA PROCESSING, CLEANING, AND MANIPULATION

# DATA PROCESSING

In [ ]:
# 1. Importing the dataset

In [28]:
import pandas as pd

healthcare = pd.read_csv(r"G:\My Drive\Data Analysis Projects\Healthcare\Dataset\patient_visits.csv")

print(healthcare.head())

  visit_id patient_id visit_date date_of_birth  patient_age patient_sex  \
0  V000001    PT00001  3/15/2022     7/22/1956           66        Male   
1  V000002    PT00002  4/22/2022     11/5/1982           39      Female   
2  V000003    PT00003  5/10/2022     2/14/1978           44        Male   
3  V000004    PT00004  1/18/2023      9/2/2004           57      Female   
4  V000005    PT00005  7/30/2023     3/19/1948           75        Male   

  icd_code  cpt_code  
0      I10     12345  
1    E11.9     23456  
2  J45.909     34567  
3    N39.0     45678  
4  I25.110     56789  


In [ ]:
# 2. Checking and addressing missing and NaN values

In [29]:
# a. Checking for missing values
healthcare.isna().sum()

visit_id         0
patient_id       0
visit_date       0
date_of_birth    0
patient_age      0
patient_sex      0
icd_code         0
cpt_code         0
dtype: int64

In [ ]:
# 3. Checking and converting data types:

In [30]:
#. Checking data types
healthcare.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   visit_id       200 non-null    object
 1   patient_id     200 non-null    object
 2   visit_date     200 non-null    object
 3   date_of_birth  200 non-null    object
 4   patient_age    200 non-null    int64 
 5   patient_sex    200 non-null    object
 6   icd_code       200 non-null    object
 7   cpt_code       200 non-null    int64 
dtypes: int64(2), object(6)
memory usage: 12.6+ KB


In [32]:
# a Converting Date columns
date_cols = ['visit_date', 'date_of_birth']
for col in date_cols:
        healthcare[col] = pd.to_datetime(healthcare[col], format='%m/%d/%Y')

In [33]:
# b. Converting to Categorical to save memory and improve speed
cat_cols = ['patient_sex']
for col in cat_cols:
    healthcare[col] = healthcare[col].astype('category')

print(healthcare.dtypes)

visit_id                 object
patient_id               object
visit_date       datetime64[ns]
date_of_birth    datetime64[ns]
patient_age               int64
patient_sex            category
icd_code                 object
cpt_code                  int64
dtype: object


In [ ]:
# 4. Checking for duplicates

In [34]:
duplicate_rows = healthcare.duplicated()
print(f"Number of duplicate rows: {duplicate_rows.sum()}")

duplicates = healthcare[healthcare.duplicated()]
print(duplicates)

Number of duplicate rows: 0
Empty DataFrame
Columns: [visit_id, patient_id, visit_date, date_of_birth, patient_age, patient_sex, icd_code, cpt_code]
Index: []


In [ ]:
# 5. Data transformation (trimming)

In [35]:
import re

# Identify object columns
obj_cols = healthcare.select_dtypes(include=['object']).columns

for col in obj_cols:
# Trimming Whitespaces
    healthcare[col] = healthcare[col].str.strip()

In [ ]:
# 6. Resetting the table's index column

In [36]:
healthcare = healthcare.reset_index(drop=True)

print(healthcare.head())
print("Cleaning Process Complete!")
print(f"Total Rows: {len(healthcare)}")
print(f"Null Values Remaining: {healthcare.isnull().sum().sum()}")

  visit_id patient_id visit_date date_of_birth  patient_age patient_sex  \
0  V000001    PT00001 2022-03-15    1956-07-22           66        Male   
1  V000002    PT00002 2022-04-22    1982-11-05           39      Female   
2  V000003    PT00003 2022-05-10    1978-02-14           44        Male   
3  V000004    PT00004 2023-01-18    2004-09-02           57      Female   
4  V000005    PT00005 2023-07-30    1948-03-19           75        Male   

  icd_code  cpt_code  
0      I10     12345  
1    E11.9     23456  
2  J45.909     34567  
3    N39.0     45678  
4  I25.110     56789  
Cleaning Process Complete!
Total Rows: 200
Null Values Remaining: 0


# DATA MANIPULATION

In [ ]:
# Feature Engineering:

In [37]:
# 1. Creating Age Bins

import pandas as pd

# Defining the bin edges and the labels for each group
# Bins: [0-18), [18-35), [35-55), [55-75), [75-120]
bins = [0, 18, 35, 55, 75, 120]
labels = ['0-17', '18-34', '35-54', '55-74', '75+']

# Creating the new age_group column
healthcare['age_group'] = pd.cut(healthcare['patient_age'], bins=bins, labels=labels, right=False)

In [38]:
# 2. Translating ICD and CPT Codes

# Mapping for common ICD-10 Diagnosis Codes 
icd_mapping = {
    'I10': 'Essential (primary) hypertension',
    'E11.9': 'Type 2 diabetes mellitus without complications',
    'J45.909': 'Unspecified asthma, uncomplicated',
    'N39.0': 'Urinary tract infection, site not specified',
    'I25.110': 'ASHD of native coronary artery with unstable angina',
    'F32.9': 'Major depressive disorder, single episode, unspecified',
    'M54.5': 'Low back pain',
    'K21.9': 'Gastro-esophageal reflux disease without esophagitis',
    'E78.5': 'Hyperlipidemia, unspecified',
    'F41.1': 'Generalized anxiety disorder',
    'E11.65': 'Type 2 diabetes mellitus with hyperglycemia',
    'J11.1': 'Influenza with respiratory manifestations',
    'M19.90': 'Unspecified osteoarthritis, unspecified site',
    'I48.91': 'Unspecified atrial fibrillation',
    'N20.0': 'Calculus of kidney',
    'G43.909': 'Migraine, unspecified, not intractable',
    'Z00.00': 'General adult medical exam without abnormal findings',
    'E66.9': 'Obesity, unspecified',
    'I25.10': 'ASHD of native coronary artery without angina pectoris',
    'I21.9': 'Acute myocardial infarction, unspecified',
    'A41.9': 'Sepsis, unspecified organism',
    'D64.9': 'Anemia, unspecified',
    'I50.9': 'Heart failure, unspecified',
    'N18.9': 'Chronic kidney disease, unspecified',
    'Z00.129': 'Routine child health exam without abnormal findings',
    'E03.9': 'Hypothyroidism, unspecified',
    'C50.912': 'Malignant neoplasm of left female breast',
    'J06.9': 'Acute upper respiratory infection, unspecified',
    'F41.9': 'Anxiety disorder, unspecified',
    'H52.1': 'Hypermetropia',
    'I64': 'Stroke, not specified as hemorrhage or infarction',
    'I64.9': 'Stroke, not specified as hemorrhage or infarction',
    'C50.911': 'Malignant neoplasm of right female breast',
    'N80.0': 'Endometriosis of uterus',
    'M17.11': 'Unilateral primary osteoarthritis, right knee',
    'J02.9': 'Acute pharyngitis, unspecified',
    'R53.1': 'Weakness',
    'M81.0': 'Age-related osteoporosis',
    'C43.9': 'Malignant melanoma of skin, unspecified',
    'K35.80': 'Unspecified acute appendicitis',
    'N418': 'Other inflammatory diseases of prostate',
    'M25.50': 'Pain in unspecified joint',
    'G47.00': 'Insomnia, unspecified',
    'E78.0': 'Pure hypercholesterolemia',
    'C34.91': 'Malignant neoplasm of upper lobe, right lung',
    'M19.9': 'Osteoarthritis, unspecified',
    'Z00.121': 'Routine child health exam with abnormal findings',
    'K80.20': 'Calculus of gallbladder with acute cholecystitis',
    'G40.909': 'Epilepsy, unspecified, not intractable',
    'I11.9': 'Hypertensive heart disease without heart failure',
    'I44.1': 'Atrioventricular block, second degree',
    'C34.90': 'Malignant neoplasm of unspecified bronchus or lung',
    'K35.30': 'Acute appendicitis with localized peritonitis',
    'M62.838': 'Other muscle spasm',
    'R10.9': 'Unspecified abdominal pain'
}



# Applying the translation
healthcare['icd_description'] = healthcare['icd_code'].map(icd_mapping).fillna('Other Diagnosis')

In [39]:
# Mapping for CPT Procedure/Service Codes
# Includes standard evaluation and management (E/M) and common procedures
cpt_mapping = {
    99211: 'Office visit, established patient, minimal',
    99212: 'Office visit, established patient, 10-19 mins',
    99213: 'Office visit, established patient, 20-29 mins',
    99214: 'Office visit, established patient, 30-39 mins',
    99215: 'Office visit, established patient, 40-54 mins',
    99201: 'Office visit, new patient, 10-19 mins',
    99202: 'Office visit, new patient, 15-29 mins',
    99203: 'Office visit, new patient, 30-44 mins',
    99204: 'Office visit, new patient, 45-59 mins',
    36415: 'Venipuncture (blood draw)',
    93000: 'ECG, routine with at least 12 leads',
    90791: 'Psychiatric diagnostic evaluation',
    97140: 'Manual therapy techniques (each 15 mins)',
    99456: 'Work-related or medical disability examination',
    92587: 'Evoked otoacoustic emissions evaluation',
    94010: 'Spirometry (vital capacity/flow rate)',
    99499: 'Unlisted evaluation and management service',
    71105: 'Radiologic examination, ribs, unilateral',
    99409: 'Alcohol/substance abuse screening (>30 mins)',
    44120: 'Enterectomy, resection of small intestine',
    31234: 'Nasal/sinus endoscopy, diagnostic',
    41002: 'Drainage of abscess from floor of mouth',
    99395: 'Preventive medicine, established patient (18-39 yrs)',
    99283: 'Emergency department visit (moderate severity)',
    99232: 'Subsequent hospital care, per day',
    90870: 'Electroconvulsive therapy',
    64612: 'Chemodenervation of muscle(s); muscle(s) innervated by facial nerve',
    99457: 'Remote physiologic monitoring treatment (first 20 mins)',
    93000: 'Electrocardiogram, routine',
    71003: 'Radiologic examination, chest, special views',
    82007: 'Ketone body(s), qualitative',
    # Placeholder/Internal codes found in data are mapped as "Internal/Test Code"
}

# Applying the translation
healthcare['cpt_description'] = pd.to_numeric(healthcare['cpt_code'], errors='coerce').map(cpt_mapping).fillna('Other Procedure/Service')

# Convert CPT to numeric to ensure mapping matches properly
healthcare['cpt_code_numeric'] = pd.to_numeric(healthcare['cpt_code'], errors='coerce')
healthcare['cpt_description'] = healthcare['cpt_code_numeric'].map(cpt_mapping).fillna('Internal/Other Procedure')

In [40]:
# 6. Organizing and Viewing Data
cols = ['visit_id', 'patient_id', 'visit_date', 'date_of_birth', 'patient_age', 'age_group', 
        'patient_sex', 'icd_code', 'icd_description', 'cpt_code', 'cpt_description']
healthcare = healthcare[cols]

# Display the result
healthcare.head(10)

,visit_id,patient_id,visit_date,date_of_birth,patient_age,age_group,patient_sex,icd_code,icd_description,cpt_code,cpt_description
0,V000001,PT00001,2022-03-15,1956-07-22,66,55-74,Male,I10,Essential (primary) hypertension,12345,Internal/Other Procedure
1,V000002,PT00002,2022-04-22,1982-11-05,39,35-54,Female,E11.9,Type 2 diabetes mellitus without complications,23456,Internal/Other Procedure
2,V000003,PT00003,2022-05-10,1978-02-14,44,35-54,Male,J45.909,"Unspecified asthma, uncomplicated",34567,Internal/Other Procedure
3,V000004,PT00004,2023-01-18,2004-09-02,57,55-74,Female,N39.0,"Urinary tract infection, site not specified",45678,Internal/Other Procedure
4,V000005,PT00005,2023-07-30,1948-03-19,75,75+,Male,I25.110,ASHD of native coronary artery with unstable a...,56789,Internal/Other Procedure
5,V000006,PT00006,2023-11-12,1988-06-21,35,35-54,Female,F32.9,"Major depressive disorder, single episode, uns...",67890,Internal/Other Procedure
6,V000007,PT00007,2024-02-08,1992-08-15,31,18-34,Female,M54.5,Low back pain,78901,Internal/Other Procedure
7,V000008,PT00008,2024-03-23,1970-12-01,53,35-54,Male,K21.9,Gastro-esophageal reflux disease without esoph...,89012,Internal/Other Procedure
8,V000009,PT00009,2022-09-14,1960-04-28,62,55-74,Female,E78.5,"Hyperlipidemia, unspecified",90123,Internal/Other Procedure
9,V000010,PT00010,2022-12-07,1985-05-30,37,35-54,Male,F41.1,Generalized anxiety disorder,91234,Internal/Other Procedure


# CREATING AND SAVING A CLEAN CSV FILE

In [41]:
healthcare.to_csv(r"G:\My Drive\Data Analysis Projects\Healthcare\Cleaned\healthcare.csv", index=False)

In [42]:
import os
file_path = r"G:\My Drive\Data Analysis Projects\Healthcare\Cleaned\healthcare.csv"
if os.path.exists(file_path):
    print("File successfully created!")

File successfully created!


# CREATING AND SAVING DATABASE

In [44]:
import pandas as pd
import sqlite3


# 1. Establishing a connection
# This creates the file 'restaurant_inspections.db' in the working directory
with sqlite3.connect('healthcare.db') as conn:

# 2. Writing the data to a table
# 'name' is the table name for the db
# 'if_exists' replaces the table if it already exists
    healthcare.to_sql(name='healthcare', con=conn, if_exists='replace', index=False)

# 3. To make sure the database is correctly built before downloading it
import os
file_stats = os.stat('healthcare.db')
print(f"Database size: {file_stats.st_size / (1024 * 1024):.2f} MB")

Database size: 0.04 MB
